# Data Cleaning

In [1]:
import pandas as pd

In [2]:
#Read raw crop data frame
crop_df = pd.read_csv("../data/Raw_Crop_df")

In [3]:
#Read weather data frame
weather_df = pd.read_csv("../data/final_county_weather_df")

In [4]:
#Read soil df
soil_df = pd.read_csv("../data/final_soil_data")

### Clean Crop Data

In [5]:
crop_df.head()

,class_desc,country_name,unit_desc,commodity_desc,group_desc,statisticcat_desc,prodn_practice_desc,freq_desc,county_ansi,CV (%),...,location_desc,watershed_desc,Value,sector_desc,domain_desc,agg_level_desc,end_code,asd_code,util_practice_desc,load_time
0,ALL CLASSES,UNITED STATES,BU / ACRE,CORN,FIELD CROPS,YIELD,ALL PRODUCTION PRACTICES,ANNUAL,NaN,NaN,...,"TENNESSEE, DELTA",NaN,131,CROPS,TOTAL,AGRICULTURAL DISTRICT,0,10.0,GRAIN,2012-01-01 00:00:00.000
1,ALL CLASSES,UNITED STATES,BU / ACRE,CORN,FIELD CROPS,YIELD,ALL PRODUCTION PRACTICES,ANNUAL,NaN,NaN,...,"TENNESSEE, WEST TENNESSEE",NaN,109,CROPS,TOTAL,AGRICULTURAL DISTRICT,0,20.0,GRAIN,2012-01-01 00:00:00.000
2,ALL CLASSES,UNITED STATES,BU / ACRE,CORN,FIELD CROPS,YIELD,ALL PRODUCTION PRACTICES,ANNUAL,NaN,NaN,...,"TENNESSEE, WESTERN RIM",NaN,106,CROPS,TOTAL,AGRICULTURAL DISTRICT,0,30.0,GRAIN,2012-01-01 00:00:00.000
3,ALL CLASSES,UNITED STATES,BU / ACRE,CORN,FIELD CROPS,YIELD,ALL PRODUCTION PRACTICES,ANNUAL,NaN,NaN,...,"TENNESSEE, CENTRAL BASIN",NaN,102,CROPS,TOTAL,AGRICULTURAL DISTRICT,0,40.0,GRAIN,2012-01-01 00:00:00.000
4,ALL CLASSES,UNITED STATES,BU / ACRE,CORN,FIELD CROPS,YIELD,ALL PRODUCTION PRACTICES,ANNUAL,NaN,NaN,...,"TENNESSEE, CUMBERLAND PLATEAU",NaN,125,CROPS,TOTAL,AGRICULTURAL DISTRICT,0,50.0,GRAIN,2012-01-01 00:00:00.000


In [6]:
#Keep only required columns
crop_df = crop_df[["county_name", "year", "commodity_desc", "Value"]].copy()

In [7]:
crop_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5384 entries, 0 to 5383
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   county_name     4497 non-null   object
 1   year            5384 non-null   int64 
 2   commodity_desc  5384 non-null   object
 3   Value           5384 non-null   object
dtypes: int64(1), object(3)
memory usage: 168.4+ KB


In [8]:
# Convert 'Value' to numeric
crop_df["Value"] = pd.to_numeric(crop_df["Value"], errors="coerce")

In [9]:
#Drop the null 'county' rows and rename it
crop_df = crop_df.dropna(subset=["county_name"])

crop_df = crop_df.rename(columns={"county_name":"county", "commodity_desc" : "crop_type", "Value":"yield_bu_acre"})

In [10]:
crop_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4497 entries, 6 to 5372
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   county         4497 non-null   object 
 1   year           4497 non-null   int64  
 2   crop_type      4497 non-null   object 
 3   yield_bu_acre  4497 non-null   float64
dtypes: float64(1), int64(1), object(2)
memory usage: 175.7+ KB


In [13]:
# Check if there are any duplicates
crop_df.duplicated(["county", "year", "crop_type"]).sum()

np.int64(641)

The Dataset contains 641 rows of duplicates

In [17]:
# Sorting the values to check patterns of duplicates 
crop_df[crop_df.duplicated(
        ["county", "year", "crop_type"],
        keep=False
    )
].sort_values(
    ["county", "year", "crop_type"]
)

,county,year,crop_type,yield_bu_acre
3691,BEDFORD,2000,WHEAT,46.0
3754,BEDFORD,2000,WHEAT,46.0
3820,BEDFORD,2001,WHEAT,52.0
3884,BEDFORD,2001,WHEAT,52.0
3956,BEDFORD,2002,WHEAT,43.0
...,...,...,...,...
4354,WILSON,2005,WHEAT,46.0
4416,WILSON,2006,WHEAT,42.0
4473,WILSON,2006,WHEAT,42.0
4542,WILSON,2007,WHEAT,30.0


In [18]:
# To make it one yield per county taking average of yield value.
crop_df = (
    crop_df
    .groupby(
        ["county", "year", "crop_type"],
        as_index=False
    )["yield_bu_acre"]
    .mean()
)

In [19]:
crop_df.duplicated(["county", "year", "crop_type"]).sum()

np.int64(0)

In [20]:
crop_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3856 entries, 0 to 3855
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   county         3856 non-null   object 
 1   year           3856 non-null   int64  
 2   crop_type      3856 non-null   object 
 3   yield_bu_acre  3856 non-null   float64
dtypes: float64(1), int64(1), object(2)
memory usage: 120.6+ KB


In [21]:
# Make consistent 'county' names.
crop_df["county"] = crop_df["county"].str.strip().str.title()

### Clean weather data

In [11]:
weather_df.head()

,county,year,PRCP,TMAX,TMIN
0,Anderson,2000,4.056389,68.944444,46.452778
1,Anderson,2001,3.903194,67.506944,44.579167
2,Anderson,2002,5.138889,69.755556,47.697222
3,Anderson,2003,6.021086,68.825253,47.759343
4,Anderson,2004,5.718889,69.255556,47.877778


In [24]:
# The dataset contains some null values for PRCP, TMAX and TMIN. Filling those values by median
weather_df["PRCP"] = weather_df["PRCP"].fillna(weather_df["PRCP"].median())
weather_df["TMAX"] = weather_df["TMAX"].fillna(weather_df["TMAX"].median())
weather_df["TMIN"] = weather_df["TMIN"].fillna(weather_df["TMIN"].median())

In [26]:
weather_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2315 entries, 0 to 2314
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   county  2315 non-null   object 
 1   year    2315 non-null   int64  
 2   PRCP    2315 non-null   float64
 3   TMAX    2315 non-null   float64
 4   TMIN    2315 non-null   float64
dtypes: float64(3), int64(1), object(1)
memory usage: 90.6+ KB


In [32]:
# Make consistent 'county' names.
weather_df["county"] = weather_df["county"].str.strip().str.title()

In [37]:
weather_df.shape

(2315, 5)

#### Clean Soil Data

In [12]:
soil_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 95 entries, 0 to 94
Data columns (total 32 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   county                                       95 non-null     object 
 1   aws025wta                                    95 non-null     float64
 2   aws050wta                                    95 non-null     float64
 3   aws0100wta                                   95 non-null     float64
 4   aws0150wta                                   95 non-null     float64
 5   slopegraddcp                                 95 non-null     float64
 6   slopegradwta                                 95 non-null     float64
 7   brockdepmin                                  95 non-null     float64
 8   wtdepannmin                                  95 non-null     float64
 9   wtdepaprjunmin                               95 non-null     float64
 10  hydg

In [30]:
# Rename Column names to descriptive columns
soil_df = soil_df.rename(columns={
    "aws025wta": "water_storage_0_25in",
    "aws050wta": "water_storage_0_50in",
    "aws0100wta": "water_storage_0_100in",
    "aws0150wta": "water_storage_0_150in",

    "slopegraddcp": "typical_slope_steepness",
    "slopegradwta": "mean_slope_weighted",
    "brockdepmin": "bedrock_depth_min",
    "wtdepannmin": "water_table_depth_avg",
    "wtdepaprjunmin": "spring_water_table_depth",

    "ph1to1h2o_r_weighted": "soil_ph_avg",
    "om_r_weighted": "organic_matter_avg",

    "drclassdcd_Excessively drained": "drainage_excessive",
    "drclassdcd_Moderately well drained": "drainage_moderate",
    "drclassdcd_Poorly drained": "drainage_poor",
    "drclassdcd_Somewhat excessively drained": "drainage_somewhat_excessive",
    "drclassdcd_Somewhat poorly drained": "drainage_somewhat_poor",
    "drclassdcd_Very poorly drained": "drainage_very_poor",
    "drclassdcd_Well drained": "drainage_well",

    "drclasswettest_Excessively drained": "wettest_excessive",
    "drclasswettest_Moderately well drained": "wettest_moderate",
    "drclasswettest_Poorly drained": "wettest_poor",
    "drclasswettest_Somewhat excessively drained": "wettest_somewhat_excessive",
    "drclasswettest_Somewhat poorly drained": "wettest_somewhat_poor",
    "drclasswettest_Very poorly drained": "wettest_very_poor",
    "drclasswettest_Well drained": "wettest_well",

    "hydgrpdcd_A": "drain_A_high",
    "hydgrpdcd_B": "drain_B_mod",
    "hydgrpdcd_C": "drain_C_low",
    "hydgrpdcd_D": "drain_D_very_low",
    "hydgrpdcd_B/D": "drain_BD",
    "hydgrpdcd_C/D": "drain_CD"
})

In [31]:
soil_df

,county,water_storage_0_25in,water_storage_0_50in,water_storage_0_100in,water_storage_0_150in,slope_degree_pct,weighted_slope_pct,bedrock_depth_min,water_table_depth_avg,spring_water_table_depth,...,drainage_well,wettest_excessive,wettest_moderate,wettest_poor,wettest_somewhat_excessive,wettest_somewhat_poor,wettest_very_poor,wettest_well,soil_ph_avg,organic_matter_avg
0,Anderson,3.156171,5.772640,9.153774,11.978131,28.770325,28.367397,23.298704,7.506619,2.767103,...,0.875085,3.461202e-07,0.069925,0.000525,0.016732,0.009593,0.000000,0.869346,4.960718,1.122919
1,Bedford,4.025817,7.452896,13.134498,16.623633,9.777086,9.769806,60.060881,15.654474,3.823853,...,0.751918,0.000000e+00,0.117521,0.024722,0.043160,0.056672,0.000000,0.751933,5.777047,1.057655
2,Benton,3.828079,7.269015,12.165571,16.159500,12.640946,12.038437,32.082159,17.723619,17.107790,...,0.426720,3.175618e-03,0.384051,0.017688,0.024201,0.042626,0.002896,0.428553,4.697170,0.559756
3,Bledsoe,3.599513,6.704664,10.377442,12.448807,18.632131,18.588435,47.961418,3.658212,0.341449,...,0.776495,0.000000e+00,0.022260,0.016520,0.180254,0.003014,0.000000,0.776545,4.944893,0.775276
4,Blount,3.506793,6.482429,10.747538,13.605522,27.598879,27.577185,38.113855,11.118141,1.426487,...,0.705347,5.161192e-02,0.044212,0.013774,0.141022,0.003621,0.000368,0.703523,5.014249,1.174345
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,Wayne,3.660509,6.887943,11.485673,13.968746,24.076832,24.073039,41.201017,18.374558,6.568320,...,0.423603,2.380442e-02,0.058815,0.009630,0.477190,0.001681,0.000000,0.425699,5.138845,0.755963
91,Weakley,5.143637,10.182476,17.794614,24.902745,6.505649,6.384514,0.000000,39.361288,30.671412,...,0.131440,0.000000e+00,0.611133,0.151997,0.000000,0.118240,0.000000,0.109781,5.237561,0.638213
92,White,4.185618,7.948731,14.435112,20.102835,15.341319,15.305502,37.115269,5.729512,3.374284,...,0.767901,0.000000e+00,0.061745,0.012865,0.054347,0.010040,0.000000,0.848876,5.141860,1.110322
93,Williamson,3.866429,7.265528,13.329327,18.355470,11.422060,11.424394,27.316191,15.667238,3.096398,...,0.630336,0.000000e+00,0.114334,0.013048,0.146225,0.013583,0.000000,0.694000,5.669861,0.980134


In [33]:
# Make consistent 'county' names.
soil_df["county"] = soil_df["county"].str.strip().str.title()

### Merge Crop data, Weather data and soil data

In [40]:
#Merge crop dataframe and weather dataframe
crop_weather_df = crop_df.merge(weather_df,on = ["county","year"], how="left")

In [43]:
# Merge crop weather data frame to soil dataframe
crop_yield_df = crop_weather_df.merge(soil_df, on ="county", how = "left")

In [44]:
crop_yield_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3856 entries, 0 to 3855
Data columns (total 38 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   county                       3856 non-null   object 
 1   year                         3856 non-null   int64  
 2   crop_type                    3856 non-null   object 
 3   yield_bu_acre                3856 non-null   float64
 4   PRCP                         3651 non-null   float64
 5   TMAX                         3651 non-null   float64
 6   TMIN                         3651 non-null   float64
 7   water_storage_0_25in         3750 non-null   float64
 8   water_storage_0_50in         3750 non-null   float64
 9   water_storage_0_100in        3750 non-null   float64
 10  water_storage_0_150in        3750 non-null   float64
 11  slope_degree_pct             3750 non-null   float64
 12  weighted_slope_pct           3750 non-null   float64
 13  bedrock_depth_min 